# hdrs

> Headers and footer that put the registry's stylesheet, script and icons into an app's pages, inline or served from content-hashed URLs

In [ ]:
#| default_exp hdrs

To use answerui in an app, pass `reg.hdrs()` and `reg.ftrs()` to `FastHTML` or `fast_app`. The headers are the stylesheet and the behavior script. The footer is the sprite sheet for icons. The headers are lazy: for each response, FastHTML renders them from the registry's current state, including any rule registered after the app was created.

In [ ]:
from fastcore.utils import *
from fasthtml.common import Style, Script, Link
from answerui.registry import Registry
import answerui.behaviors  # installs Registry.build_js
from fastlucide.icons import spritesheet
import hashlib, mimetypes
from starlette.responses import Response

In [ ]:
from fasthtml.common import *
from answerui.registry import reg, hypelt
from fastcore.test import *
from fastlucide.icons import Check
from tempfile import TemporaryDirectory

`assets` caches the stylesheet and script, because building the stylesheet visits every rule. Rule registration and removal increment `version`. `assets` rebuilds when `version` or `htmx4` changes:

In [ ]:
@patch
def assets(self:Registry):
    "The stylesheet and script, keyed by extension, built once per registry version"
    key = self.version, self.htmx4
    if getattr(self, '_assets_version', None) != key:
        self._assets = {'css': self.build_css(), 'js': self.build_js()}
        self._assets_version = key
    return self._assets

`assets` returns the same build until a rule changes:

In [ ]:
r = Registry()
built = r.assets()
test_is(r.assets(), built)
r.rule('main { max-width: 65ch; }')
assert r.assets() is not built
assert 'max-width: 65ch' in r.assets()['css']
list(built)

'ee68738f88fc'

Changing `htmx4` rebuilds the script while keeping the CSS unchanged:

In [ ]:
htmx2 = r.assets()
r.htmx4 = True
htmx4 = r.assets()
test_eq(htmx4['css'], htmx2['css'])
assert htmx4['js'] != htmx2['js']

When FastHTML renders a page, it calls `__ft__` on each object that has one and renders the result in its place. `_Lazy` wraps a function in an object whose `__ft__` calls it. `hdrs` returns the stylesheet and behavior script as two `_Lazy` objects. FastHTML renders them again for every response. Because `hdrs` belongs to the registry, an app with its own `Registry` gets that registry's stylesheet and script by passing its `hdrs()` in place of `reg.hdrs()`:

In [ ]:
class _Lazy:
    def __init__(self, f): self.f = f
    def __ft__(self): return self.f()

@patch
def hdrs(self:Registry):
    "Headers for `fast_app(hdrs=...)`: the stylesheet and behavior script, rendered from this registry per response"
    return _Lazy(lambda: Style(self.assets()['css'])), _Lazy(lambda: Script(self.assets()['js']))


To see that the headers are lazy, build them first and define a component afterwards. Rendering the stylesheet header then includes the component's CSS.

In [ ]:
hdrs = reg.hdrs()
LateCard = hypelt('LateCard', css='display: block; color: teal;', layer='components',
    on={('&', 'click'): r"el.remove()"}, doc='Defined after the app; click to remove.')
css = to_xml(hdrs[0])
print(css[css.index('/* LateCard.base */'):][:60])

/* LateCard.base */
late-card { display: block; color: teal;


Redefining the component changes the next render. There is no rebuild step and nothing to invalidate:

In [ ]:
LateCard = hypelt('LateCard', css='display: block; color: navy;', layer='components', doc='Defined after the app.')
css = to_xml(hdrs[0])
assert 'navy' in css and 'teal' not in css

An icon is a reference to a symbol in fastlucide's `spritesheet`. When a module imports an icon from `fastlucide.icons`, the name is added to the sheet. The sheet is an `svg` element and belongs in the body, not the head. `ftrs` returns the sheet for FastHTML to render at the end of the body. `spritesheet` has an `__ft__` method that renders the sheet with each page. `ftrs` therefore returns it without a `_Lazy` wrapper:

In [ ]:
@patch
def ftrs(self:Registry):
    "Footers for `fast_app(ftrs=...)`: fastlucide's sprite sheet, with every icon that any module has imported"
    return (spritesheet,)

## A whole page

That is all an app needs. FastHTML's test client fetches a page without starting a server. The head of the response has htmx, then our stylesheet and script. The sprite sheet is at the end of the body, after the icon that uses it.

In [ ]:
app = FastHTML(hdrs=reg.hdrs(), ftrs=reg.ftrs())
rt = app.route

@rt
def index(): return Titled('Hello', LateCard(Check(), 'Defined after the app.'))

page = Client(app).get('/').text
assert page.index('<symbol id="lc-check"') > page.index('late-card')
print(re.sub(r'(?s)(<(style|script)>).*?(?=</\2>)', r'\1…', page))

<!doctype html>
<html>
  <head>
    <title>Hello</title>
    <link rel="canonical" href="https://testserver/">
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"></script><script src="https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.13/fasthtml.js"></script><script src="https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js"></script><script src="https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js"></script>    <style>…</style>
<script>…</script>  </head>
  <body>
<main class="container">      <h1>Hello</h1>
<late-card><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-check"></use></svg>Defined after the app.</late-card></main>    <style>…</style>
<svg xmlns="http://www.w3.org/2000/svg" style="display: none"><defs><symbol id="lc-check"><path d="M20

`fast_app` works the same way, but it adds Pico CSS by default, so we turn that off.

In [ ]:
app, rt = fast_app(hdrs=reg.hdrs(), ftrs=reg.ftrs(), pico=False)
@rt
def index(): return Titled('Hello', LateCard('Defined after the app.'))

assert 'pico' not in (r:= Client(app).get('/').text)
assert 'late-card' in r

For HTMX 4, set `reg.htmx4 = True` before calling `reg.hdrs()` and create the app with `FastHTML(htmx4=reg.htmx4)`. `reg.serve(app)` copies `app.htmx4` into the registry during setup. Configure the version before building headers or previews.

## Serving the assets

With inline headers, every response includes the whole stylesheet and script. `serve` instead mounts them at URLs named by a hash of their content. The browser fetches each once and caches it. The headers that `serve` adds to the app are lazy, like those from `hdrs`. While the registry is unchanged, every page links to the same names. When a rule changes, the next page links to a new name. The browser then fetches the new file. With several worker processes, a page and its assets can come from different workers. Each worker builds and serves the current assets on request, without first rendering a page. Requests for unknown names don't cause rebuilds, because `serve` names the build of each registry version only once. `serve` remembers every build it has linked or served. For a current or remembered name, its route returns that build, marked immutable. For any other name it returns 404, even when another worker built that version. Because browsers and proxies keep an immutable response for a year, a name must never return different content. Each name is a hash of exactly the content served under it. `serve` also adds the sprite sheet from `ftrs()` to the app's footers. It returns the two headers, for an app that builds its own `head`.

In [ ]:
def _hash(s): return hashlib.sha256(s.encode()).hexdigest()[:12]

@patch
def serve(
    self:Registry,
    app,              # The FastHTML app to mount the route on and add the headers to
    path='/answerui', # URL prefix the assets are served under
):
    "Serve the stylesheet and script under content-hashed names, adding their headers and the sprite sheet to `app`; returns the two headers, for an app that builds its own head"
    self.htmx4 = app.htmx4
    served, names, mimes = {}, {}, {'css': 'text/css', 'js': 'text/javascript'}
    def local(body):
        "`body` with each registered file's URL replaced by the URL of its served copy"
        for name,(p,url) in self.files.items():
            data = p.read_bytes()
            fname = f'{hashlib.sha256(data).hexdigest()[:12]}-{name}'
            served[fname] = (data, mimetypes.guess_type(name)[0])
            body = body.replace(url, f'{path}/{fname}')
        return body
    def link(ext):
        key = ext, self.version, self.htmx4
        if key not in names:
            body = local(self.assets()[ext])
            names[key] = name = f'{_hash(body)}.{ext}'
            served[name] = (body, mimes[ext])
        return f'{path}/{names[key]}'
    @app.get(f'{path}/{{name}}')
    def asset(name: str):
        for ext in mimes: link(ext)
        if name not in served: return Response(status_code=404)
        body, mime = served[name]
        return Response(body, media_type=mime, headers={'Cache-Control': 'public, max-age=31536000, immutable'})
    hdrs = _Lazy(lambda: Link(rel='stylesheet', href=link('css'))), _Lazy(lambda: Script(src=link('js')))
    app.hdrs, app.ftrs = [*app.hdrs, *hdrs], [*app.ftrs, *self.ftrs()]
    return hdrs

In [ ]:
app, worker = FastHTML(), FastHTML()
rt = app.route
for a in (app, worker): reg.serve(a)

@rt
def index(): return Titled('Hello', LateCard('Defined after the app.'))

page = Client(app).get('/').text
href = re.search(r'href="(/answerui/[^"]+\.css)"', page)[1]
asset = Client(worker).get(href)
test_eq(asset.headers['cache-control'], 'public, max-age=31536000, immutable')
assert 'navy' in asset.text
href

'/answerui/523580fc6bd8.css'

After a change to the registry, the next page links to a new name. The old name still returns the build it was linked with, and an unknown name returns 404:

In [ ]:
LateCard = hypelt('LateCard', css='display: block; color: teal;', layer='components', doc='Defined after the app.')
href2 = re.search(r'href="(/answerui/[^"]+\.css)"', Client(app).get('/').text)[1]
assert href2 != href
assert 'navy' in Client(app).get(href).text
assert 'teal' in Client(app).get(href2).text
test_eq(Client(app).get('/answerui/000000000000.css').status_code, 404)

Some rules need a file that isn't part of the stylesheet, such as a font. Register the file with `reg.file`, giving a name to serve it under, its path on disk and a public URL. Use the URL that `reg.file` returns in the rule. `serve` serves the file from the app and replaces the public URL in the served stylesheet with the file's served URL. Without `serve`, the browser fetches the file from the public URL.


In [ ]:
r, tmp = Registry(), TemporaryDirectory()
logo = Path(tmp.name)/'logo.svg'
logo.write_text('<svg xmlns="http://www.w3.org/2000/svg"/>')
url = r.file('logo.svg', logo, 'https://example.com/logo.svg')
r.rule(f"header {{ background: url('{url}'); }}")
assert url in r.build_css()

The served stylesheet's name is a hash of its content, including the copy's URL. Another worker can serve the app's copy of the file without first rendering a page:

In [ ]:
app, worker = FastHTML(), FastHTML()
for a in (app, worker): r.serve(a)

@app.get('/')
def index(): return Titled('Hello')

cli = Client(app)
css_href = re.search(r'href="(/answerui/[^"]+\.css)"', cli.get('/').text)[1]
css = cli.get(css_href).text
test_eq(css_href, f'/answerui/{_hash(css)}.css')
assert url not in css
served_logo = re.search(r"url\('(/answerui/\w+-logo\.svg)'\)", css)[1]
test_eq(Client(worker).get(served_logo).text, logo.read_text())
served_logo


## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()